# code 1

In [1]:
import os
import numpy as np
import pandas as pd
from Bio import Phylo


# ============================================================
# Normalized Robinson-Foulds distance
# ============================================================

def get_leaf_names(tree):
    """
    Return the set of leaf names in a Bio.Phylo tree.
    """
    return {terminal.name for terminal in tree.get_terminals()}


def get_splits(tree, leaf_names):
    """
    Extract non-trivial bipartitions/splits from a tree.

    For an unrooted RF comparison, a split and its complement are treated
    as the same split. Therefore, we store the smaller side of the split.
    """
    splits = set()
    all_leaves = set(leaf_names)
    n = len(all_leaves)

    for clade in tree.find_clades(order="level"):
        clade_leaves = {terminal.name for terminal in clade.get_terminals()}

        # Ignore trivial splits
        if len(clade_leaves) <= 1 or len(clade_leaves) >= n - 1:
            continue

        complement = all_leaves - clade_leaves

        # Store the smaller side to make split/complement equivalent
        if len(clade_leaves) <= len(complement):
            split = frozenset(clade_leaves)
        else:
            split = frozenset(complement)

        splits.add(split)

    return splits


def normalized_rf(treefile1, treefile2):
    """
    Compute normalized Robinson-Foulds distance between two Newick trees.

    The two trees must have the same leaf names. If they have different
    leaf sets, the function reports the mismatch and stops.

    Normalized RF is computed as:

        nRF = RF / (number of splits in tree1 + number of splits in tree2)

    where

        RF = number of splits present in one tree but not the other.
    """

    # Read trees
    tree1 = Phylo.read(treefile1, "newick")
    tree2 = Phylo.read(treefile2, "newick")

    leaves1 = get_leaf_names(tree1)
    leaves2 = get_leaf_names(tree2)

    if leaves1 != leaves2:
        only_in_tree1 = sorted(leaves1 - leaves2)
        only_in_tree2 = sorted(leaves2 - leaves1)

        print("\nLeaf sets do not match.")
        print("Leaves only in tree 1:", only_in_tree1)
        print("Leaves only in tree 2:", only_in_tree2)

        raise ValueError("The two trees must have the same leaf names.")

    leaf_names = leaves1

    splits1 = get_splits(tree1, leaf_names)
    splits2 = get_splits(tree2, leaf_names)

    shared_splits = splits1.intersection(splits2)

    rf = len(splits1 - splits2) + len(splits2 - splits1)
    max_rf = len(splits1) + len(splits2)

    if max_rf == 0:
        nrf = 0.0
    else:
        nrf = rf / max_rf

    return nrf


# ============================================================
# Desired k-values for the reported nRF computations
# ============================================================

desired_k = {
    "ebolavirus_record": 4,
    "HEV_record": 4,
    "influenzaHAgene_record": 4,
    "mammalianMT_record": 4,
    "rhinovirus_record": 5,
    "sarscov2_record": 6,
}


# ============================================================
# MAFFT reference trees
# ============================================================

mafft_tree_map = {
    "ebolavirus_record": "./trees2/mafft/ebolavirus_UPGMA_relabelled_accessions.nwk",
    "HEV_record": "./trees2/mafft/HEV_UPGMA_relabelled_accessions.nwk",
    "influenzaHAgene_record": "./trees2/mafft/influenza_UPGMA_relabelled_accessions.nwk",
    "mammalianMT_record": "./trees2/mafft/mammalian_UPGMA_relabelled_accessions.nwk",
    "rhinovirus_record": "./trees2/mafft/rhinovirus_UPGMA_relabelled_accessions.nwk",
    "sarscov2_record": "./trees2/mafft/sars_UPGMA_relabelled_accessions.nwk",
}


# ============================================================
# Datasets to process
# ============================================================

trees = [
    "ebolavirus_record",
    "HEV_record",
    "influenzaHAgene_record",
    "mammalianMT_record",
    "rhinovirus_record",
    "sarscov2_record",
]


# ============================================================
# Compute nRF for each dataset and average
# ============================================================

results = []

for test in trees:
    k = desired_k[test]

    tree_a = f"./trees2/CAKR/{test}/k{k}_tree_facet0.txt"
    tree_b = mafft_tree_map[test]

    print("\n==============================")
    print(f"Dataset: {test}")
    print(f"k = {k}")
    print("CAKR tree:", tree_a)
    print("MAFFT tree:", tree_b)

    if not os.path.exists(tree_a):
        print("Missing CAKR tree:", tree_a)
        continue

    if not os.path.exists(tree_b):
        print("Missing MAFFT tree:", tree_b)
        continue

    nrf = normalized_rf(tree_a, tree_b)

    print("Normalized RF distance:", nrf)
    print("Normalized RF distance rounded:", round(nrf, 2))

    results.append({
        "Dataset": test,
        "k": k,
        "CAKR_tree": tree_a,
        "MAFFT_tree": tree_b,
        "nRF": nrf,
        "nRF_rounded_2": round(nrf, 2),
    })


# ============================================================
# Results table
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n==============================")
print("Final nRF results")
print("==============================")

print(results_df[["Dataset", "k", "nRF", "nRF_rounded_2"]])


# ============================================================
# Average nRF
# ============================================================

if len(results_df) > 0:
    average_nrf = results_df["nRF"].mean()

    print("\nAverage nRF:", average_nrf)
    print("Average nRF rounded to 2 decimals:", round(average_nrf, 2))
    print("Average nRF rounded to 3 decimals:", round(average_nrf, 3))

    average_row = {
        "Dataset": "Average",
        "k": "",
        "CAKR_tree": "",
        "MAFFT_tree": "",
        "nRF": average_nrf,
        "nRF_rounded_2": round(average_nrf, 2),
    }

    results_df_with_average = pd.concat(
        [results_df, pd.DataFrame([average_row])],
        ignore_index=True
    )

else:
    print("\nNo nRF values were computed.")
    results_df_with_average = results_df


# ============================================================
# Save output
# ============================================================

results_df_with_average.to_csv("nrf_results.csv", index=False)

print("\nSaved file:")
print("nrf_results.csv")


Dataset: ebolavirus_record
k = 4
CAKR tree: ./trees2/CAKR/ebolavirus_record/k4_tree_facet0.txt
MAFFT tree: ./trees2/mafft/ebolavirus_UPGMA_relabelled_accessions.nwk
Normalized RF distance: 0.4107142857142857
Normalized RF distance rounded: 0.41

Dataset: HEV_record
k = 4
CAKR tree: ./trees2/CAKR/HEV_record/k4_tree_facet0.txt
MAFFT tree: ./trees2/mafft/HEV_UPGMA_relabelled_accessions.nwk
Normalized RF distance: 0.37777777777777777
Normalized RF distance rounded: 0.38

Dataset: influenzaHAgene_record
k = 4
CAKR tree: ./trees2/CAKR/influenzaHAgene_record/k4_tree_facet0.txt
MAFFT tree: ./trees2/mafft/influenza_UPGMA_relabelled_accessions.nwk
Normalized RF distance: 0.2
Normalized RF distance rounded: 0.2

Dataset: mammalianMT_record
k = 4
CAKR tree: ./trees2/CAKR/mammalianMT_record/k4_tree_facet0.txt
MAFFT tree: ./trees2/mafft/mammalian_UPGMA_relabelled_accessions.nwk
Normalized RF distance: 0.42105263157894735
Normalized RF distance rounded: 0.42

Dataset: rhinovirus_record
k = 5
CAKR tr